# Investigation 4 — Event-derived CN and antecedent state

**Participant-directed investigation.** Use observed rainfall and
direct-runoff events to infer event curve numbers, estimate an
asymptotic response, and compare two antecedent-state conventions.

**Minimum result:** one fitted curve number reported with lambda, event
count, model form, and diagnostic, plus one comparison of rainfall-
history and root-zone-wetness classifications.


In [ ]:
# V3 portable setup: local repository, GitHub Pages bundle, or Colab.
from pathlib import Path
import hashlib
import importlib
import importlib.util
import os
import subprocess
import sys
import urllib.request
import zipfile

CNKIT_VERSION = "1.1.0"
BUNDLE_URL = (
    "https://skp703.github.io/cn-workshop-2026/"
    "downloads/cn_workshop_v3_data.zip"
)
BUNDLE_SHA256 = "925861246fe9520c4b7f399227ca6133e60f27a5063a73c9fb6715cd9904780c"


def _is_workshop_root(path):
    return (path / "data" / "sites.csv").exists() and (path / "prepared").exists()


def _find_workshop_root():
    candidates = [
        Path.cwd(),
        Path.cwd().parent,
        Path("/content/cnkit_workshop"),
    ]
    requested = os.environ.get("CNKIT_WORKSHOP_HOME")
    if requested:
        candidates.insert(0, Path(requested).expanduser())
    for candidate in candidates:
        candidate = candidate.resolve()
        if _is_workshop_root(candidate):
            return candidate, "existing workshop folder"

    destination = Path("/content/cnkit_workshop") if Path("/content").exists() else Path.cwd() / ".cnkit_workshop"
    destination.mkdir(parents=True, exist_ok=True)
    archive = destination / "cn_workshop_v3_data.zip"
    print("Downloading the versioned V3 workshop bundle...")
    request = urllib.request.Request(BUNDLE_URL, headers={"User-Agent": "cn-workshop-v3"})
    with urllib.request.urlopen(request, timeout=120) as response, archive.open("wb") as handle:
        handle.write(response.read())
    digest = hashlib.sha256(archive.read_bytes()).hexdigest()
    if digest != BUNDLE_SHA256:
        raise RuntimeError(
            "Workshop bundle checksum mismatch. Expected %s, received %s. "
            "Delete %s and try again." % (BUNDLE_SHA256, digest, archive)
        )
    with zipfile.ZipFile(archive) as zipped:
        zipped.extractall(destination)
    if not _is_workshop_root(destination):
        raise RuntimeError("The workshop bundle downloaded but required files are missing.")
    return destination.resolve(), "checksum-verified workshop download"


WORKSHOP_ROOT, DATA_SOURCE = _find_workshop_root()
DATA_DIR = WORKSHOP_ROOT / "data"
PREPARED_DIR = WORKSHOP_ROOT / "prepared"


def _load_cnkit():
    try:
        import cnkit as package
        if getattr(package, "__version__", None) == CNKIT_VERSION:
            return package, "installed package"
    except ImportError:
        pass

    for candidate in [
        WORKSHOP_ROOT / "vendor" / "cnkit.py",
        WORKSHOP_ROOT / "cnkit.py",
        Path.cwd() / "vendor" / "cnkit.py",
        Path.cwd().parent / "vendor" / "cnkit.py",
    ]:
        if candidate.exists():
            spec = importlib.util.spec_from_file_location("cnkit", candidate)
            package = importlib.util.module_from_spec(spec)
            sys.modules["cnkit"] = package
            spec.loader.exec_module(package)
            return package, str(candidate)

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit==" + CNKIT_VERSION]
    )
    importlib.invalidate_caches()
    import cnkit as package
    return package, "PyPI"


def activate_full_cnkit():
    """Return the installed package with data, delineation, and GEE modules."""
    global cnkit, CNKIT_SOURCE
    if hasattr(cnkit, "__path__"):
        return cnkit
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", "cnkit[gee]==" + CNKIT_VERSION]
    )
    for name in [key for key in sys.modules if key == "cnkit" or key.startswith("cnkit.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    cnkit = importlib.import_module("cnkit")
    CNKIT_SOURCE = "PyPI with Earth Engine dependencies"
    return cnkit


cnkit, CNKIT_SOURCE = _load_cnkit()
print("cnkit version:", getattr(cnkit, "__version__", CNKIT_VERSION + " workshop module"))
print("cnkit source :", CNKIT_SOURCE)
print("data source  :", DATA_SOURCE)
print("data folder  :", DATA_DIR)
print("setup complete")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from cnkit import (
    CN_from_PQ,
    compare_conventions,
    doy_climatology,
    fit_asymptotic,
    runoff,
    sm_percentile,
)

DESIGN_DEPTH_IN = 4.78


## Part 1 — Curve numbers inferred from observed events

This part changes the evidence base from spatial lookup tables to
rainfall and direct-runoff observations at streamgages.


## Step 1 — Invert the rainfall–runoff equation

For an event with measured rainfall $P$ and direct-runoff depth $Q$,
`CN_from_PQ` solves the curve-number equation backward. With
$I_a=\lambda S$, the physically admissible root is used to recover
$S$, followed by

$$
CN=\frac{1000}{S+10}.
$$

Event CN is therefore a transformed observation, not a direct sensor
measurement. It depends on rainfall, hydrograph separation and runoff
volume, watershed area, event definition, and the selected lambda.
Events with $Q\leq0$, $Q>P$, or no physically valid solution are
excluded from the asymptotic fit.


In [ ]:
difficult_events = pd.read_csv(
    DATA_DIR / "events_01646000.csv", parse_dates=["start", "end"]
)
difficult_events["CN_lambda_020"] = CN_from_PQ(
    difficult_events.P_in.values,
    difficult_events.Q_in.values,
    lam=0.20,
)
difficult_events["CN_lambda_005"] = CN_from_PQ(
    difficult_events.P_in.values,
    difficult_events.Q_in.values,
    lam=0.05,
)
display(
    difficult_events[
        ["start", "P_in", "Q_in", "runoff_ratio", "CN_lambda_020", "CN_lambda_005"]
    ].head(10).round(3)
)
print("event records:", len(difficult_events))


## Step 2 — Estimate the standard asymptotic response

Event-derived CN commonly varies with storm depth. The Hawkins
standard response represents a decreasing sequence that approaches a
stable value as rainfall increases:

$$
CN(P)=CN_{\infty}+(100-CN_{\infty})e^{-kP}.
$$

`fit_asymptotic` first derives event CN with the specified lambda, then
uses bounded nonlinear least squares to estimate $CN_{\infty}$ and
$k$. It returns the fitted parameters, event count, RMSE, and
coefficient of determination. The diagnostic statistics describe this
functional fit; they do not account for uncertainty in precipitation,
discharge, or hydrograph separation.


In [ ]:
fits = []
fitted_objects = {}
for watershed, gage, table_cn in [
    ("Difficult Run", "01646000", 75.5),
    ("Accotink Creek", "01654000", 77.9),
]:
    events = pd.read_csv(DATA_DIR / ("events_" + gage + ".csv"))
    fit20 = fit_asymptotic(events.P_in.values, events.Q_in.values, lam=0.20)
    fit05 = fit_asymptotic(events.P_in.values, events.Q_in.values, lam=0.05)
    fitted_objects[(gage, 0.20)] = fit20
    fitted_objects[(gage, 0.05)] = fit05
    fits.append(
        {
            "watershed": watershed,
            "events": len(events),
            "table_CN": table_cn,
            "CN_inf_lambda_020": fit20.cn_inf,
            "r2_lambda_020": fit20.r2,
            "CN_inf_lambda_005": fit05.cn_inf,
            "r2_lambda_005": fit05.r2,
        }
    )
fit_table = pd.DataFrame(fits).set_index("watershed")
display(fit_table.round(3))


In [ ]:
difficult_fit = fitted_objects[("01646000", 0.20)]
valid_event_cn = difficult_events.replace([np.inf, -np.inf], np.nan).dropna(
    subset=["P_in", "CN_lambda_020"]
)
rainfall_grid = np.linspace(
    valid_event_cn.P_in.min(), valid_event_cn.P_in.max(), 250
)

fig, ax = plt.subplots(figsize=(8.2, 4.8))
ax.scatter(
    valid_event_cn.P_in,
    valid_event_cn.CN_lambda_020,
    s=18,
    alpha=0.32,
    color="#6f7f89",
    label="event-derived CN",
)
ax.plot(
    rainfall_grid,
    difficult_fit.predict(rainfall_grid),
    color="#c85d45",
    lw=2.8,
    label=r"standard fit, $CN_{\infty}=%.1f$" % difficult_fit.cn_inf,
)
ax.axhline(75.5, color="#007f92", ls="--", label="table CN = 75.5")
ax.set(xlabel="event rainfall, inches", ylabel="event-derived curve number")
ax.set_ylim(0, 103)
ax.grid(alpha=0.25)
ax.legend()
plt.show()


## Step 3 — Treat lambda and CN as a paired calibration

The tabulated curve numbers were developed with the conventional
relation $I_a=0.20S$. Replacing lambda with 0.05 changes both the
rainfall threshold and the fitted event CN. The two fitted columns
above show why a reported CN must include its lambda; the number and
the equation convention form one calibration.

Compare the two fitted $CN_{\infty}$ values and their diagnostics.
A better fit under one lambda is evidence about this event sample, not
a universal conversion factor for another watershed.


In [ ]:
gage_names = {
    "01646000": "Difficult Run",
    "01654000": "Accotink Creek",
}
paired_rows = []
for (gage, lam), fitted in fitted_objects.items():
    paired_rows.append(
        {
            "watershed": gage_names[gage],
            "lambda": lam,
            "CN_infinity": fitted.cn_inf,
            "events_fitted": fitted.n_events,
            "R_squared": fitted.r2,
            "runoff_at_design_depth_in": float(
                runoff(DESIGN_DEPTH_IN, fitted.cn_inf, lam=lam)
            ),
        }
    )

paired_calibration = pd.DataFrame(paired_rows).sort_values(
    ["watershed", "lambda"], ascending=[True, False]
)
display(paired_calibration.round(3))


**Interpretation.** Compare lambda values within one watershed. The
fitted CN changes because the inverse event equation changes, while the
final runoff column places each fitted CN back inside its corresponding
equation. Report the CN, lambda, event count, and fit diagnostic as one
calibration record.


## Part 2 — Antecedent-condition conventions

This part compares two operational descriptions of the watershed state
before an event: five-day rainfall history and seasonally standardized
root-zone wetness.


## Step 4 — Distinguish rainfall history from observed wetness

The historical antecedent moisture condition (AMC) convention assigns
class I, II, or III from five-day rainfall thresholds that vary between
growing and dormant seasons. NEH-630 now uses the broader term
antecedent runoff condition (ARC) to emphasize that runoff response
also reflects cover, temperature, frozen ground, and event history.

The alternative examined here uses NASA POWER `GWETROOT`, a
model-assimilated root-zone wetness index. It represents a broad soil
layer at a comparatively coarse spatial scale; it is not an in-situ
soil-moisture measurement for every point in the watershed.

Raw wetness values have a seasonal cycle. `doy_climatology` pools all
observations within ±15 calendar days of each day of year across the
record. `sm_percentile` compares the previous day's value with that
local seasonal pool. This makes a January and July percentile
comparable while retaining the stated 31-day window as an analytical
choice.


In [ ]:
events = pd.read_csv(DATA_DIR / "events_01646000.csv", parse_dates=["start"])
precipitation = pd.read_csv(
    DATA_DIR / "precip_01646000.csv", parse_dates=["date"]
).set_index("date").P_in
moisture = pd.read_csv(
    DATA_DIR / "soilmoisture_power_01646000.csv", parse_dates=["date"]
).set_index("date").GWETROOT

climatology = doy_climatology(moisture, window=15)
events["previous_day"] = events.start.dt.normalize() - pd.Timedelta(days=1)
events["root_zone_wetness"] = events.previous_day.map(moisture)
events["wetness_percentile"] = [
    sm_percentile(moisture, day, climatology=climatology)
    for day in events.previous_day
]
events["event_cn"] = CN_from_PQ(
    events.P_in.values, events.Q_in.values, lam=0.20
)

display(
    events[
        ["start", "P_in", "Q_in", "root_zone_wetness", "wetness_percentile", "event_cn"]
    ].head(10).round(3)
)


## Step 5 — Compare the conventions on the same storm dates

`compare_conventions` performs both classifications without blending
them. For every event date it:

1. sums the preceding five days of daily rainfall and applies the
   historical seasonal AMC thresholds;
2. calculates the previous-day wetness percentile from the ±15-day
   climatology;
3. maps each class to a CN relative to the same fair-condition `cn2`;
4. optionally applies each CN to the same design storm.

The disagreement rate is a sensitivity diagnostic: it identifies how
often the two proxies point to different antecedent states.


In [ ]:
convention_comparison = compare_conventions(
    events.start,
    precipitation,
    moisture,
    cn2=75.5,
    design_depth_in=DESIGN_DEPTH_IN,
    window=15,
)

print("comparable events:", convention_comparison.attrs["n_comparable"])
print("agreements:       ", convention_comparison.attrs["n_agree"])
print("disagreement rate: %.3f" % convention_comparison.attrs["disagreement_rate"])
display(
    pd.crosstab(
        convention_comparison.AMC_direction,
        convention_comparison.SM_direction,
        margins=True,
    )
)
display(
    convention_comparison.loc[
        ~convention_comparison.conventions_agree,
        [
            "date", "P5_in", "AMC_direction", "SM_percentile",
            "SM_direction", "CN_from_AMC", "CN_from_SM",
            "Q_from_AMC_in", "Q_from_SM_in",
        ],
    ].head(12)
)


## Step 6 — Relate wetness rank to observed event response

The percentile record can also be divided into equal-width lower,
middle, and upper ranges. Grouping observed event CN and runoff ratio
by those ranges tests whether wetter antecedent states correspond to a
systematically different event response in this record. The groups are
descriptive; they do not redefine NRCS ARC classes.


In [ ]:
valid = events.replace([np.inf, -np.inf], np.nan).dropna(
    subset=["wetness_percentile", "event_cn", "runoff_ratio"]
).copy()
valid["wetness_range"] = pd.cut(
    valid.wetness_percentile,
    bins=[0, 33.333, 66.667, 100],
    labels=["lower third", "middle third", "upper third"],
    include_lowest=True,
)
antecedent_summary = valid.groupby("wetness_range", observed=True).agg(
    events=("event_cn", "size"),
    median_percentile=("wetness_percentile", "median"),
    median_event_cn=("event_cn", "median"),
    median_runoff_ratio=("runoff_ratio", "median"),
)
display(antecedent_summary.round(3))


## Open investigation — Choose a question

1. Compare lambda 0.20 and 0.05 within one watershed and retain the
   fitted event count and diagnostic.
2. Compare Difficult Run and Accotink Creek using the same model and
   event-screening rules.
3. Remove the smallest rainfall events and evaluate the stability of
   $CN_{\infty}$.
4. Change the wetness-climatology window and examine the convention
   disagreement rate.
5. Test whether observed event CN or runoff ratio differs systematically
   among wetness ranges.


## Method audit and reporting record

| Layer | Library operation | Analyst responsibility |
|---|---|---|
| `core` | Evaluates and inverts the rainfall–runoff equation | Lambda, event definition, rainfall and runoff basis |
| `asymptotic` | Fits the selected response model to event-derived CN | Model family, event screening, and fit interpretation |
| `antecedent` | Calculates five-day rainfall and wetness-percentile conventions side by side | Proxy, climatology window, thresholds, and interpretation |

Record: watershed and event period; equation convention; fitted model,
event count, parameter, and diagnostic; antecedent proxies and window;
extension result; interpretation; and limitation.

## References and data sources

- Hawkins, R. H. 1993. “Asymptotic Determination of Runoff Curve
  Numbers from Data.” *Journal of Irrigation and Drainage Engineering*
  119(2):334–345.
  [doi:10.1061/(ASCE)0733-9437(1993)119:2(334)](https://doi.org/10.1061/%28ASCE%290733-9437%281993%29119%3A2%28334%29).
- Woodward, D. E., R. H. Hawkins, R. Jiang, A. T. Hjelmfelt Jr.,
  J. A. Van Mullem, and Q. D. Quan. 2003. “Runoff Curve Number Method:
  Examination of the Initial Abstraction Ratio.” *World Water &
  Environmental Resources Congress 2003*, 1–10.
  [doi:10.1061/40685(2003)308](https://doi.org/10.1061/40685%282003%29308).
- U.S. Geological Survey. [Water Data for the Nation](https://waterdata.usgs.gov/nwis/),
  [doi:10.5066/F7P55KJN](https://doi.org/10.5066/F7P55KJN).
- PRISM Group, Oregon State University. [PRISM climate data](https://prism.oregonstate.edu/terms/),
  accessed 9 August 2026 through the
  [ACIS web service](https://docs.rcc-acis.org/acisws/).
- NASA POWER. [Daily API documentation](https://power.larc.nasa.gov/docs/services/api/temporal/daily/).
- USDA NRCS. 2004. [NEH Part 630, Chapter 10](https://directives.nrcs.usda.gov/sites/default/files2/1712930608/7300.pdf).

The complete cross-notebook source ledger is available in
[workshop source ledger](https://github.com/skp703/cn-workshop-2026/blob/main/docs/SOURCES.md).
